
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Lecture - MLflow's Evaluation Framework
#### Duration: 20 minutes

## Overview

MLflow provides a comprehensive evaluation framework specifically designed for generative AI applications, offering automated judges, tracing capabilities, and systematic assessment tools. This lecture explores MLflow's architecture, core components, and how they work together to enable rigorous agent evaluation.

We'll examine the three fundamental components of MLflow evaluation, understand the `mlflow.genai.evaluate()` function, and explore how tracing provides the foundation for sophisticated evaluation capabilities.

## Learning Objectives

By the end of this lecture, you will be able to:
- Describe MLflow's evaluation framework architecture and key components
- Understand the role of evaluation datasets, scorers, and predict functions
- Explain how the `mlflow.genai.evaluate()` function orchestrates evaluation
- Understand the role of tracing in agent evaluation and debugging
- Recognize how production monitoring and Unity Gateway extend evaluation to a deployed agent

## A. MLflow for Agent Evaluation

### A1. MLflow for Agent Evaluation

During development, an **evaluation harness** helps you quickly execute your application for every record in your evaluation set and then run each output through your LLM judges and metric computations. In this module, we focus on the evaluation-specific capabilities that are purpose-built for assessing AI agents using Databricks AI Agent Evaluation.

The five MLflow capabilities you will use most for agent evaluation are **tracing**, **scorers and judges**, **`mlflow.genai.evaluate()`**, **evaluation experiments** for comparing runs, and **production monitoring**.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<strong style="color: #0d47a1; font-size: 1.0em;">OpenTelemetry Compatible</strong>
<p style="margin: 8px 0 0 0; color: #333; font-size: 14px;">MLflow traces are fully compatible with OpenTelemetry specs, enabling export to external observability platforms (Datadog, Grafana, Splunk, etc.) via MLflow-only, OTel-only, or dual export modes.</p>
<p style="margin: 6px 0 0 0; font-size: 13px;">Docs: (<a href="https://docs.databricks.com/aws/en/mlflow3/genai/tracing/integrations/open-telemetry/" target="_blank">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/mlflow3/genai/tracing/integrations/open-telemetry/" target="_blank">Azure</a> | <a href="https://docs.databricks.com/gcp/en/mlflow3/genai/tracing/integrations/open-telemetry/" target="_blank">GCP</a>)</p>
</div>

## B. Core Architecture



### B1. Three Fundamental Components

MLflow's evaluation architecture centers on three components that flow into `mlflow.genai.evaluate()`:

<div style="background: #fff; max-width: 1600px; margin: 0 auto; padding: 12px 0 24px;">
<svg viewBox="0 0 1600 532.6500000000001" width="100%" role="img" aria-label="Overview of mlflow.genai.evaluate(): an evaluation dataset, scorers, and a predict function feed into evaluate(), which calls the agent, traces each call, runs the scorers, and logs the run; it returns an EvaluationResult with a run_id and aggregated metrics." style="display: block; height: auto; font-family: -apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Arial,sans-serif;">
<defs>
<marker id="eo-ar" viewBox="0 0 10 10" refX="8.5" refY="5" markerWidth="8" markerHeight="8" orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#618794"/></marker>
<filter id="eo-sh" x="-10%" y="-10%" width="120%" height="130%"><feDropShadow dx="0" dy="3" stdDeviation="6" flood-color="#1B3139" flood-opacity="0.08"/></filter>
</defs>
<rect x="40" y="20" width="440" height="148" rx="14" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#eo-sh)"/>
<rect x="62" y="34" width="396" height="5" rx="2.5" fill="#2272B4"/>
<rect x="62" y="53" width="188" height="26" rx="13.0" fill="#2272B4"/><text x="156" y="71" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">EVALUATION DATASET</text>
<text x="62" y="112" font-size="21" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">What to test</text>
<text x="62" y="146" font-size="19" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">inputs, expectations, outputs</text>
<rect x="40" y="187.55" width="440" height="148" rx="14" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#eo-sh)"/>
<rect x="62" y="201.55" width="396" height="5" rx="2.5" fill="#2272B4"/>
<rect x="62" y="221" width="88" height="26" rx="13.0" fill="#2272B4"/><text x="106" y="238" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">SCORERS</text>
<text x="62" y="280" font-size="21" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">How to grade</text>
<text x="62" y="313" font-size="19" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">built-in, guideline, custom, code</text>
<rect x="40" y="355.1" width="440" height="148" rx="14" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#eo-sh)"/>
<rect x="62" y="369.1" width="396" height="5" rx="2.5" fill="#2272B4"/>
<rect x="62" y="388" width="170" height="26" rx="13.0" fill="#2272B4"/><text x="147" y="406" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">PREDICT FUNCTION</text>
<text x="62" y="447" font-size="21" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">How to call the agent</text>
<text x="62" y="481" font-size="19" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">optional for pre-generated outputs</text>
<rect x="580" y="160.00000000000006" width="450" height="203" rx="14" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#eo-sh)"/>
<rect x="602" y="174.00000000000006" width="406" height="5" rx="2.5" fill="#FF5F46"/>
<rect x="602" y="193" width="133" height="26" rx="13.0" fill="#FF5F46"/><text x="669" y="211" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">ORCHESTRATES</text>
<text x="602" y="252" font-size="21" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">mlflow.genai.evaluate()</text>
<text x="602" y="286" font-size="19" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">Calls the agent for each row, traces every</text>
<text x="602" y="313" font-size="19" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">call, runs each scorer, and logs the run</text>
<text x="602" y="341" font-size="19" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">to MLflow.</text>
<rect x="1120" y="160.00000000000006" width="440" height="203" rx="14" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#eo-sh)"/>
<rect x="1142" y="174.00000000000006" width="396" height="5" rx="2.5" fill="#1B5162"/>
<rect x="1142" y="193" width="88" height="26" rx="13.0" fill="#1B5162"/><text x="1186" y="211" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">RETURNS</text>
<text x="1142" y="252" font-size="21" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">EvaluationResult</text>
<text x="1142" y="286" font-size="19" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">A run_id and aggregated metrics such as</text>
<text x="1142" y="313" font-size="19" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">correctness/mean. Per-row assessments</text>
<text x="1142" y="341" font-size="19" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">live on each row's trace.</text>
<path d="M484,93.775 H520 Q540,93.775 540,113.775 V241.32500000000005 Q540,261.32500000000005 560,261.32500000000005 H574" fill="none" stroke="#618794" stroke-width="2.4" marker-end="url(#eo-ar)"/>
<path d="M484,261.32500000000005 H574" fill="none" stroke="#618794" stroke-width="2.4" marker-end="url(#eo-ar)"/>
<path d="M484,428.875 H520 Q540,428.875 540,408.875 V281.32500000000005 Q540,261.32500000000005 560,261.32500000000005 H574" fill="none" stroke="#618794" stroke-width="2.4" marker-end="url(#eo-ar)"/>
<path d="M1034,261.32500000000005 H1114" fill="none" stroke="#618794" stroke-width="2.4" marker-end="url(#eo-ar)"/>
</svg>
</div>

<details style="margin: 14px 0; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 0 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07); font-family: sans-serif;">
<summary style="cursor: pointer; padding: 16px 0; font-weight: 700; font-size: 18px; color: #0b2026;">Evaluation Dataset</summary>
<div style="padding: 0 0 18px 0; font-size: 16px; color: #5A6F77; line-height: 1.6;">
Your evaluation dataset defines what you're testing. Each record in an evaluation dataset represents a single test case with the following structure:
<ul style="margin:8px 0;">
<li><strong>inputs</strong> (required): The test input data that will be passed to your model or application</li>
<li><strong>outputs</strong> (optional): The actual outputs generated by your model, typically used for post-hoc evaluation</li>
<li><strong>expectations</strong> (optional): The expected outputs or quality criteria that define correct behavior (e.g., expected facts, expected responses, or per-row guidelines)</li>
<li><strong>source</strong> (optional): Provenance information about how this record was created</li>
<li><strong>tags</strong> (optional): Metadata specific to this individual record for organization and filtering</li>
</ul>
Datasets are typically stored as JSON files or Pandas DataFrames for easy manipulation and versioning.
</div>
</details>

<details style="margin: 14px 0; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 0 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07); font-family: sans-serif;">
<summary style="cursor: pointer; padding: 16px 0; font-weight: 700; font-size: 18px; color: #0b2026;">Scorers (Judges)</summary>
<div style="padding: 0 0 18px 0; font-size: 16px; color: #5A6F77; line-height: 1.6;">
Scorers evaluate your agent's outputs against defined criteria. Choose the right type depending on how much customization and control you need:
<ul style="margin:8px 0;">
<li><strong>Built-in judges</strong> - Research-validated LLM-based assessments for common quality dimensions (Correctness, Safety, RelevanceToQuery, RetrievalGroundedness, etc.). This category also includes <strong>Guidelines</strong> and <strong>ExpectationsGuidelines</strong> judges that check whether responses pass or fail custom natural-language rules.</li>
<li><strong>Custom LLM judges</strong> - Fully customized LLM judges with detailed evaluation criteria and feedback optimization. Capable of returning numerical scores, categories, or boolean values.</li>
<li><strong>Code-based scorers</strong> - Programmatic and deterministic scorers for exact matching, format validation, and performance metrics.</li>
<li><strong>Third-party scorers</strong> - Specialized metrics from open-source evaluation frameworks.</li>
</ul>
Start with built-in judges for quick evaluation. As your needs evolve, build custom LLM judges for domain-specific criteria and create custom code-based scorers for deterministic business logic.
</div>
</details>

<details style="margin: 14px 0; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 0 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07); font-family: sans-serif;">
<summary style="cursor: pointer; padding: 16px 0; font-weight: 700; font-size: 18px; color: #0b2026;">Predict Function</summary>
<div style="padding: 0 0 18px 0; font-size: 16px; color: #5A6F77; line-height: 1.6;">
The predict function generates outputs for your evaluation dataset. This can be:
<ul style="margin:8px 0;">
<li>Your agent's prediction method (for evaluating on-the-fly)</li>
<li>A lambda that transforms inputs to the format your agent expects</li>
<li>Omitted entirely if you're evaluating pre-generated outputs ("answer sheet" mode)</li>
</ul>
These three components come together in <code>mlflow.genai.evaluate()</code>, which orchestrates the evaluation process, collects metrics, and logs comprehensive results for analysis.
</div>
</details>

## C. The `mlflow.genai.evaluate()` Function


### C1. Central Orchestration Point

The `mlflow.genai.evaluate()` function provides an **evaluation harness**: a structured way to feed in test data, run your app, and automatically score the results.

<details style="margin: 14px 0; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 0 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07); font-family: sans-serif;">
<summary style="cursor: pointer; padding: 16px 0; font-weight: 700; font-size: 18px; color: #0b2026;">How it works</summary>
<div style="padding: 0 0 18px 0; font-size: 16px; color: #5A6F77; line-height: 1.6;">
Instead of manually running your app and checking outputs one by one, MLflow Evaluation connects offline testing with production monitoring: the same evaluation logic you use in development can also run in production.

<pre style="background:#f8f8f8;border:1px solid #e0e0e0;border-radius:6px;padding:12px;margin:10px 0;font-size:13px;overflow-x:auto;">
import mlflow
from mlflow.genai.scorers import Correctness

results = mlflow.genai.evaluate(
    data=eval_dataset,              # EvaluationDataset, DataFrame, list[dict], etc.
    scorers=[Correctness()],        # Built-in and/or custom scorers
    predict_fn=my_app,              # Optional: direct evaluation
    # model_id="m-abc123...",        # Optional: link to LoggedModel
)
</pre>

Docs: (<a href="https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/concepts/eval-harness/" target="_blank">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/mlflow3/genai/eval-monitor/concepts/eval-harness" target="_blank">Azure</a> | <a href="https://docs.databricks.com/gcp/en/mlflow3/genai/eval-monitor/concepts/eval-harness/" target="_blank">GCP</a>)
</div>
</details>


### C2. Key Parameters

<div style="background: #fff; max-width: 1600px; margin: 0 auto; padding: 12px 0 24px;">
<svg viewBox="0 0 1600 540" width="100%" role="img" aria-label="An annotated mlflow.genai.evaluate() call. The code panel is on the left and explanation cards are on the right. Numbered markers point to its four parts: data (what to test), predict_fn (how to call the agent), scorers (how to grade, with judges on Unity Gateway model services), and the EvaluationResult with run_id and metrics." style="display: block; height: auto; font-family: -apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Arial,sans-serif;">
<defs>
<marker id="ev-ar" viewBox="0 0 10 10" refX="8.5" refY="5" markerWidth="8" markerHeight="8" orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#618794"/></marker>
<filter id="ev-sh" x="-10%" y="-10%" width="120%" height="130%"><feDropShadow dx="0" dy="3" stdDeviation="6" flood-color="#1B3139" flood-opacity="0.08"/></filter>
</defs>
<rect x="20" y="20" width="760" height="506" rx="18" fill="#303F47" filter="url(#ev-sh)"/>
<text x="56" y="66" font-size="15" font-weight="700" fill="#8FA3AD" text-anchor="start" letter-spacing="1" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">mlflow.genai.evaluate()</text>
<text x="56" y="120" font-size="15" font-weight="400" fill="#E6EBEE" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">results = mlflow.genai.<tspan fill="#8FD6B4">evaluate</tspan>(</text>
<text x="56" y="150" font-size="15" font-weight="400" fill="#E6EBEE" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">    data=eval_dataset,</text>
<text x="56" y="180" font-size="15" font-weight="400" fill="#E6EBEE" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">    predict_fn=<tspan fill="#FF8774">lambda</tspan> input: agent.predict({<tspan fill="#FFCC66">"input"</tspan>: input}),</text>
<text x="56" y="210" font-size="15" font-weight="400" fill="#E6EBEE" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">    scorers=[</text>
<text x="56" y="240" font-size="15" font-weight="400" fill="#E6EBEE" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">        Correctness(model=<tspan fill="#FFCC66">"databricks:/workspace.agent_eval.judge-model"</tspan>),</text>
<text x="56" y="270" font-size="15" font-weight="400" fill="#E6EBEE" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">        Safety(model=<tspan fill="#FFCC66">"databricks:/workspace.agent_eval.judge-model"</tspan>),</text>
<text x="56" y="300" font-size="15" font-weight="400" fill="#E6EBEE" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">        completeness_judge,  <tspan fill="#9FB2BB"># make_judge(...)</tspan></text>
<text x="56" y="330" font-size="15" font-weight="400" fill="#E6EBEE" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">    ],</text>
<text x="56" y="360" font-size="15" font-weight="400" fill="#E6EBEE" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">)</text>
<text x="56" y="390" font-size="15" font-weight="400" fill="#E6EBEE" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve"></text>
<text x="56" y="420" font-size="15" font-weight="400" fill="#E6EBEE" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">results.run_id, results.metrics</text>
<rect x="800" y="20" width="375" height="245" rx="14" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#ev-sh)"/>
<rect x="820" y="34" width="335" height="5" rx="2.5" fill="#FF5F46"/>
<rect x="820" y="53" width="75" height="26" rx="13.0" fill="#FF5F46"/><text x="858" y="71" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">DATA</text>
<text x="820" y="102" font-size="21" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">What to test</text>
<text x="820" y="132" font-size="19" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">Rows of inputs, plus optional</text>
<text x="820" y="159" font-size="19" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">outputs and expectations: a list</text>
<text x="820" y="186" font-size="19" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">of dicts, a DataFrame, or an</text>
<text x="820" y="213" font-size="19" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">evaluation dataset in Unity</text>
<text x="820" y="240" font-size="19" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">Catalog.</text>
<rect x="1190" y="20" width="375" height="245" rx="14" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#ev-sh)"/>
<rect x="1210" y="34" width="335" height="5" rx="2.5" fill="#FF5F46"/>
<rect x="1210" y="53" width="120" height="26" rx="13.0" fill="#FF5F46"/><text x="1270" y="71" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">PREDICT_FN</text>
<text x="1210" y="102" font-size="21" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">How to call the agent</text>
<text x="1210" y="132" font-size="19" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">Called once per row with the</text>
<text x="1210" y="159" font-size="19" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">row's inputs. Leave it out to</text>
<text x="1210" y="186" font-size="19" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">score outputs that are already in</text>
<text x="1210" y="213" font-size="19" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">the data.</text>
<rect x="800" y="281" width="375" height="245" rx="14" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#ev-sh)"/>
<rect x="820" y="295" width="335" height="5" rx="2.5" fill="#FF5F46"/>
<rect x="820" y="314" width="100" height="26" rx="13.0" fill="#FF5F46"/><text x="870" y="332" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">SCORERS</text>
<text x="820" y="363" font-size="21" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">How to grade</text>
<text x="820" y="393" font-size="19" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">Built-in, guideline, custom, and</text>
<text x="820" y="420" font-size="19" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">code scorers. Each judge's model</text>
<text x="820" y="447" font-size="19" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">is a Unity Gateway model service.</text>
<rect x="1190" y="281" width="375" height="245" rx="14" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#ev-sh)"/>
<rect x="1210" y="295" width="335" height="5" rx="2.5" fill="#FF5F46"/>
<rect x="1210" y="314" width="100" height="26" rx="13.0" fill="#FF5F46"/><text x="1260" y="332" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">RESULTS</text>
<text x="1210" y="363" font-size="21" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">What you get back</text>
<text x="1210" y="393" font-size="19" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">An EvaluationResult with a run_id</text>
<text x="1210" y="420" font-size="19" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">and aggregated metrics. Per-row</text>
<text x="1210" y="447" font-size="19" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">assessments live on each row's</text>
<text x="1210" y="474" font-size="19" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">trace.</text>
</svg>
</div>

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<strong style="color: #0d47a1; font-size: 1.0em;">predict_fn requirements</strong>
<p style="margin: 8px 0 0 0; color: #333; font-size: 14px;">When provided, <code>predict_fn</code> must accept keys from the <code>inputs</code> dictionary as keyword arguments, return a JSON-serializable dictionary, be instrumented with MLflow Tracing, and emit exactly one trace per call.</p>
</div>

Docs: (<a href="https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/concepts/eval-harness/" target="_blank">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/mlflow3/genai/eval-monitor/concepts/eval-harness" target="_blank">Azure</a> | <a href="https://docs.databricks.com/gcp/en/mlflow3/genai/eval-monitor/concepts/eval-harness/" target="_blank">GCP</a>)


### C3. Evaluation Workflow

When you call `mlflow.genai.evaluate()`, it runs these steps:

1. **Load the data** and validate its structure.
2. **Generate outputs** by calling `predict_fn` for each row. This step is skipped when the data already has outputs.
3. **Create traces**: each call is traced automatically.
4. **Run the scorers** on every row.
5. **Aggregate and log** the metrics and results to MLflow.



### C4. Return Value and Results Access

<div style="background: #fff; max-width: 1600px; margin: 0 auto; padding: 12px 0 24px;">
<svg viewBox="0 0 1600 686" width="100%" role="img" aria-label="A mock of an evaluation run: aggregated metrics correctness 0.80, safety 1.00, tone 0.60, and a per-row table of the correctness questions with yes/no results for each judge. One failing correctness cell opens its rationale. Per-row results are read with mlflow.search_traces." style="display: block; height: auto; font-family: -apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Arial,sans-serif;">
<defs>
<marker id="rv-ar" viewBox="0 0 10 10" refX="8.5" refY="5" markerWidth="8" markerHeight="8" orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#618794"/></marker>
<filter id="rv-sh" x="-10%" y="-10%" width="120%" height="130%"><feDropShadow dx="0" dy="3" stdDeviation="6" flood-color="#1B3139" flood-opacity="0.08"/></filter>
</defs>
<rect x="40" y="20" width="1520" height="642" rx="14" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#rv-sh)"/>
<text x="68" y="62" font-size="15" font-weight="700" fill="#618794" text-anchor="start" letter-spacing="1.4" xml:space="preserve">EVALUATION RUN</text>
<text x="68" y="100" font-size="26" font-weight="800" fill="#0b2026" text-anchor="start" xml:space="preserve">airbnb_experiment</text>
<text x="312" y="100" font-size="17" font-weight="500" fill="#618794" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">run_id 3f9c2e…</text>
<rect x="840" y="40" width="220" height="86" rx="12" fill="#F9F7F4" stroke="#EEEDE9"/>
<rect x="858" y="54" width="184" height="5" rx="2.5" fill="#2272B4"/>
<text x="858" y="84" font-size="15" font-weight="600" fill="#618794" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">correctness/mean</text>
<text x="858" y="114" font-size="28" font-weight="800" fill="#0b2026" text-anchor="start" xml:space="preserve">0.80</text>
<rect x="1080" y="40" width="220" height="86" rx="12" fill="#F9F7F4" stroke="#EEEDE9"/>
<rect x="1098" y="54" width="184" height="5" rx="2.5" fill="#2272B4"/>
<text x="1098" y="84" font-size="15" font-weight="600" fill="#618794" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">safety/mean</text>
<text x="1098" y="114" font-size="28" font-weight="800" fill="#0b2026" text-anchor="start" xml:space="preserve">1.00</text>
<rect x="1320" y="40" width="220" height="86" rx="12" fill="#F9F7F4" stroke="#EEEDE9"/>
<rect x="1338" y="54" width="184" height="5" rx="2.5" fill="#2272B4"/>
<text x="1338" y="84" font-size="15" font-weight="600" fill="#618794" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">tone/mean</text>
<text x="1338" y="114" font-size="28" font-weight="800" fill="#0b2026" text-anchor="start" xml:space="preserve">0.60</text>
<text x="1540" y="152" font-size="15" font-weight="600" fill="#618794" text-anchor="end" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">results.metrics</text>
<rect x="56" y="172" width="1020" height="44" rx="8" fill="#F9F7F4"/>
<text x="68" y="200" font-size="15" font-weight="700" fill="#618794" text-anchor="start" letter-spacing="1.4" xml:space="preserve">INPUT</text>
<text x="650" y="200" font-size="15" font-weight="700" fill="#618794" text-anchor="start" letter-spacing="1.4" xml:space="preserve">CORRECTNESS</text>
<text x="830" y="200" font-size="15" font-weight="700" fill="#618794" text-anchor="start" letter-spacing="1.4" xml:space="preserve">SAFETY</text>
<text x="980" y="200" font-size="15" font-weight="700" fill="#618794" text-anchor="start" letter-spacing="1.4" xml:space="preserve">TONE</text>
<text x="68" y="262" font-size="18" font-weight="400" fill="#0b2026" text-anchor="start" xml:space="preserve">How many Entire home/apt listings are in the Mission?</text>
<rect x="650" y="242" width="64" height="26" rx="13.0" fill="#00A972"/><text x="682" y="260" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">YES</text>
<rect x="830" y="242" width="64" height="26" rx="13.0" fill="#00A972"/><text x="862" y="260" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">YES</text>
<rect x="980" y="242" width="64" height="26" rx="13.0" fill="#00A972"/><text x="1012" y="260" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">YES</text>
<line x1="56" y1="285" x2="1076" y2="285" stroke="#EEEDE9"/>
<text x="68" y="322" font-size="18" font-weight="400" fill="#0b2026" text-anchor="start" xml:space="preserve">Count the number of Private room listings in Nob Hill.</text>
<rect x="650" y="302" width="64" height="26" rx="13.0" fill="#00A972"/><text x="682" y="320" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">YES</text>
<rect x="830" y="302" width="64" height="26" rx="13.0" fill="#00A972"/><text x="862" y="320" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">YES</text>
<rect x="980" y="302" width="64" height="26" rx="13.0" fill="#FF5F46"/><text x="1012" y="320" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">NO</text>
<line x1="56" y1="345" x2="1076" y2="345" stroke="#EEEDE9"/>
<rect x="56" y="348" width="1020" height="54" rx="8" fill="#FFF6F4" stroke="#FABFBA"/>
<text x="68" y="382" font-size="18" font-weight="600" fill="#0b2026" text-anchor="start" xml:space="preserve">What is the average listing price in Haight Ashbury?</text>
<rect x="650" y="362" width="64" height="26" rx="13.0" fill="#FF5F46"/><text x="682" y="380" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">NO</text>
<rect x="830" y="362" width="64" height="26" rx="13.0" fill="#00A972"/><text x="862" y="380" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">YES</text>
<rect x="980" y="362" width="64" height="26" rx="13.0" fill="#00A972"/><text x="1012" y="380" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">YES</text>
<line x1="56" y1="405" x2="1076" y2="405" stroke="#EEEDE9"/>
<text x="68" y="442" font-size="18" font-weight="400" fill="#0b2026" text-anchor="start" xml:space="preserve">What's the average price in Middle Earth?</text>
<rect x="650" y="422" width="64" height="26" rx="13.0" fill="#00A972"/><text x="682" y="440" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">YES</text>
<rect x="830" y="422" width="64" height="26" rx="13.0" fill="#00A972"/><text x="862" y="440" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">YES</text>
<rect x="980" y="422" width="64" height="26" rx="13.0" fill="#00A972"/><text x="1012" y="440" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">YES</text>
<line x1="56" y1="465" x2="1076" y2="465" stroke="#EEEDE9"/>
<text x="68" y="502" font-size="18" font-weight="400" fill="#0b2026" text-anchor="start" xml:space="preserve">How many Shared room listings are in Middle Earth?</text>
<rect x="650" y="482" width="64" height="26" rx="13.0" fill="#00A972"/><text x="682" y="500" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">YES</text>
<rect x="830" y="482" width="64" height="26" rx="13.0" fill="#00A972"/><text x="862" y="500" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">YES</text>
<rect x="980" y="482" width="64" height="26" rx="13.0" fill="#FF5F46"/><text x="1012" y="500" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">NO</text>
<line x1="56" y1="525" x2="1076" y2="525" stroke="#EEEDE9"/>
<path d="M1076,375.0 H1116" fill="none" stroke="#FF5F46" stroke-width="2" stroke-dasharray="5 4"/>
<rect x="1116" y="278.525" width="430" height="193" rx="14" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#rv-sh)"/>
<rect x="1138" y="292.525" width="386" height="5" rx="2.5" fill="#FF5F46"/>
<rect x="1138" y="312" width="215" height="26" rx="13.0" fill="#FF5F46"/><text x="1246" y="329" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">CORRECTNESS RATIONALE</text>
<text x="1138" y="370" font-size="20" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">Average price in Haight Ashbury</text>
<text x="1138" y="400" font-size="17" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">The response gives an average of 92.40, but</text>
<text x="1138" y="425" font-size="17" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">the expected facts say the average price is</text>
<text x="1138" y="449" font-size="17" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">87.71.</text>
<line x1="68" y1="552" x2="1532" y2="552" stroke="#EEEDE9" stroke-width="1.5"/>
<text x="68" y="592" font-size="15" font-weight="700" fill="#618794" text-anchor="start" letter-spacing="1.4" xml:space="preserve">PER-ROW RESULTS LIVE ON THE TRACES</text>
<text x="68" y="632" font-size="17" font-weight="500" fill="#0b2026" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">traces = mlflow.search_traces(run_id=results.run_id, return_type=<tspan fill="#E8590C">"list"</tspan>)</text>
</svg>
</div>

The function returns an `EvaluationResult` object with `run_id`, `metrics` (aggregated dictionary), `artifacts`, and `tables` (per-example DataFrames). Per-example traces and feedback are accessed separately.

<details style="margin: 14px 0; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 0 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07); font-family: sans-serif;">
<summary style="cursor: pointer; padding: 16px 0; font-weight: 700; font-size: 18px; color: #0b2026;">Evaluation run structure</summary>
<div style="padding: 0 0 18px 0; font-size: 16px; color: #5A6F77; line-height: 1.6;">
Each evaluation run contains:
<ul style="margin:8px 0;">
<li><strong>Traces</strong> - One trace for each input in your evaluation dataset</li>
<li><strong>Feedback</strong> - Quality assessments from scorers attached to each trace</li>
<li><strong>Metrics</strong> - Aggregate statistics (average score, pass rate, etc.)</li>
<li><strong>Metadata</strong> - Information about the evaluation configuration</li>
</ul>
The <code>EvaluationResult</code> object provides <strong>run_id</strong> and <strong>metrics</strong> (aggregated dictionary).
</div>
</details>

<details style="margin: 14px 0; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 0 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07); font-family: sans-serif;">
<summary style="cursor: pointer; padding: 16px 0; font-weight: 700; font-size: 18px; color: #0b2026;">Accessing per-example results</summary>
<div style="padding: 0 0 18px 0; font-size: 16px; color: #5A6F77; line-height: 1.6;">
In MLflow 3, per-example results are accessed via traces:
<pre style="background:#f8f8f8;border:1px solid #e0e0e0;border-radius:6px;padding:12px;margin:10px 0;font-size:13px;overflow-x:auto;">
eval_traces = mlflow.search_traces(run_id=results.run_id, return_type="list")

for trace in eval_traces:
    assessments = trace.info.assessments
    for assessment in assessments:
        print(f"Scorer: {assessment.name}, Value: {assessment.value}")
</pre>
Docs: (<a href="https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/concepts/evaluation-runs/" target="_blank">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/mlflow3/genai/eval-monitor/concepts/evaluation-runs" target="_blank">Azure</a> | <a href="https://docs.databricks.com/gcp/en/mlflow3/genai/eval-monitor/concepts/evaluation-runs/" target="_blank">GCP</a>)
</div>
</details>

## D. MLflow Tracing: The Foundation of Agent Observability


### D1. Comprehensive Observability

MLflow Tracing provides comprehensive observability into agent execution, capturing every step of your agent's reasoning process. Tracing is enabled when using `mlflow.genai.evaluate()` with properly instrumented predict functions.

Each trace records the model calls (prompts, responses, and parameters), tool invocations (inputs and return values), retrieval operations (documents and metadata), the duration of every operation, and the parent-child hierarchy that ties them together.

Docs: (<a href="https://docs.databricks.com/aws/en/mlflow3/genai/tracing/span-concepts/" target="_blank">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/mlflow3/genai/tracing/span-concepts" target="_blank">Azure</a> | <a href="https://docs.databricks.com/gcp/en/mlflow3/genai/tracing/span-concepts/" target="_blank">GCP</a>)


### D2. Span Types in Traces

MLflow organizes traces into spans with specific types. Here is one trace from a tool-calling agent like the course's Airbnb agent, with the tool span selected:

<div style="background: #fff; max-width: 1600px; margin: 0 auto; padding: 12px 0 24px;">
<svg viewBox="0 0 1600 660" width="100%" role="img" aria-label="A trace from a tool-calling agent shown as a span waterfall: the airbnb_assistant AGENT span contains predict_stream, which contains two CHAT_MODEL spans and an avg_neigh_price TOOL span. The selected TOOL span shows its inputs, output 87.71, and latency." style="display: block; height: auto; font-family: -apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Arial,sans-serif;">
<defs>
<marker id="wf-ar" viewBox="0 0 10 10" refX="8.5" refY="5" markerWidth="8" markerHeight="8" orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#618794"/></marker>
<filter id="wf-sh" x="-10%" y="-10%" width="120%" height="130%"><feDropShadow dx="0" dy="3" stdDeviation="6" flood-color="#1B3139" flood-opacity="0.08"/></filter>
</defs>
<rect x="40" y="20" width="1060" height="620" rx="14" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#wf-sh)"/>
<text x="68" y="62" font-size="15" font-weight="700" fill="#618794" text-anchor="start" letter-spacing="1.4" xml:space="preserve">EXAMPLE TRACE · TOOL-CALLING AGENT</text>
<text x="68" y="100" font-size="24" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">What is the average listing price in Haight Ashbury?</text>
<text x="68" y="152" font-size="15" font-weight="700" fill="#618794" text-anchor="start" letter-spacing="1.4" xml:space="preserve">SPAN</text>
<text x="420" y="152" font-size="15" font-weight="700" fill="#618794" text-anchor="start" letter-spacing="1.4" xml:space="preserve">TYPE</text>
<text x="600" y="152" font-size="15" font-weight="700" fill="#618794" text-anchor="start" letter-spacing="1.4" xml:space="preserve">TIMELINE</text>
<line x1="600" y1="166" x2="600" y2="510" stroke="#EEEDE9" stroke-width="1.5"/>
<text x="600" y="534" font-size="16" font-weight="500" fill="#618794" text-anchor="middle" xml:space="preserve">0 s</text>
<line x1="725" y1="166" x2="725" y2="510" stroke="#EEEDE9" stroke-width="1.5"/>
<text x="725" y="534" font-size="16" font-weight="500" fill="#618794" text-anchor="middle" xml:space="preserve">1 s</text>
<line x1="850" y1="166" x2="850" y2="510" stroke="#EEEDE9" stroke-width="1.5"/>
<text x="850" y="534" font-size="16" font-weight="500" fill="#618794" text-anchor="middle" xml:space="preserve">2 s</text>
<line x1="975" y1="166" x2="975" y2="510" stroke="#EEEDE9" stroke-width="1.5"/>
<text x="975" y="534" font-size="16" font-weight="500" fill="#618794" text-anchor="middle" xml:space="preserve">3 s</text>
<text x="72" y="210" font-size="18" font-weight="600" fill="#0b2026" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">airbnb_assistant</text>
<rect x="420" y="190" width="70" height="26" rx="13.0" fill="#1B5162"/><text x="455" y="208" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">AGENT</text>
<rect x="600" y="191" width="400" height="24" rx="6" fill="#1B5162" fill-opacity="0.85"/>
<text x="1010" y="210" font-size="16" font-weight="500" fill="#618794" text-anchor="start" xml:space="preserve">3.20 s</text>
<path d="M84,230 V270 H96" fill="none" stroke="#C6DCEA" stroke-width="1.5"/>
<text x="102" y="276" font-size="18" font-weight="600" fill="#0b2026" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">predict_stream</text>
<rect x="420" y="256" width="70" height="26" rx="13.0" fill="#1B5162"/><text x="455" y="274" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">AGENT</text>
<rect x="605" y="257" width="390" height="24" rx="6" fill="#1B5162" fill-opacity="0.85"/>
<text x="1005" y="276" font-size="16" font-weight="500" fill="#618794" text-anchor="start" xml:space="preserve">3.12 s</text>
<path d="M114,296 V336 H126" fill="none" stroke="#C6DCEA" stroke-width="1.5"/>
<text x="132" y="342" font-size="18" font-weight="600" fill="#0b2026" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">chat.completions</text>
<rect x="420" y="322" width="115" height="26" rx="13.0" fill="#2272B4"/><text x="478" y="340" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">CHAT_MODEL</text>
<rect x="610" y="323" width="162" height="24" rx="6" fill="#2272B4" fill-opacity="0.85"/>
<text x="782" y="342" font-size="16" font-weight="500" fill="#618794" text-anchor="start" xml:space="preserve">1.30 s</text>
<rect x="56" y="372" width="1030" height="58" rx="10" fill="#FFF6F4" stroke="#FABFBA"/>
<path d="M114,362 V402 H126" fill="none" stroke="#C6DCEA" stroke-width="1.5"/>
<text x="132" y="408" font-size="18" font-weight="700" fill="#0b2026" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">avg_neigh_price</text>
<rect x="420" y="388" width="60" height="26" rx="13.0" fill="#00A972"/><text x="450" y="406" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">TOOL</text>
<rect x="778" y="389" width="57" height="24" rx="6" fill="#00A972" fill-opacity="1"/>
<text x="845" y="408" font-size="16" font-weight="500" fill="#618794" text-anchor="start" xml:space="preserve">0.46 s</text>
<path d="M114,428 V468 H126" fill="none" stroke="#C6DCEA" stroke-width="1.5"/>
<text x="132" y="474" font-size="18" font-weight="600" fill="#0b2026" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">chat.completions</text>
<rect x="420" y="454" width="115" height="26" rx="13.0" fill="#2272B4"/><text x="478" y="472" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">CHAT_MODEL</text>
<rect x="840" y="455" width="148" height="24" rx="6" fill="#2272B4" fill-opacity="0.85"/>
<text x="998" y="474" font-size="16" font-weight="500" fill="#618794" text-anchor="start" xml:space="preserve">1.18 s</text>
<text x="68" y="600" font-size="17" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">Spans nest like a call tree. Select one to see its inputs, outputs, and timing.</text>
<rect x="1130" y="20" width="430" height="620" rx="14" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#wf-sh)"/>
<rect x="1152" y="34" width="386" height="5" rx="2.5" fill="#FF5F46"/>
<rect x="1152" y="53" width="142" height="26" rx="13.0" fill="#FF5F46"/><text x="1223" y="71" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">SELECTED SPAN</text>
<text x="1152" y="120" font-size="22" font-weight="700" fill="#0b2026" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">avg_neigh_price</text>
<rect x="1152" y="136" width="60" height="26" rx="13.0" fill="#00A972"/><text x="1182" y="154" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">TOOL</text>
<text x="1152" y="202" font-size="15" font-weight="700" fill="#618794" text-anchor="start" letter-spacing="1.4" xml:space="preserve">INPUTS</text>
<rect x="1152" y="214" width="386" height="50" rx="8" fill="#303F47"/>
<text x="1168" y="246" font-size="16" font-weight="400" fill="#E6EBEE" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">{"neighborhood_name": <tspan fill="#FFCC66">"Haight Ashbury"</tspan>}</text>
<text x="1152" y="300" font-size="15" font-weight="700" fill="#618794" text-anchor="start" letter-spacing="1.4" xml:space="preserve">OUTPUTS</text>
<rect x="1152" y="312" width="386" height="50" rx="8" fill="#303F47"/>
<text x="1168" y="344" font-size="16" font-weight="400" fill="#E6EBEE" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">87.71  <tspan fill="#8FD6B4">-- average listing price</tspan></text>
<text x="1152" y="398" font-size="15" font-weight="700" fill="#618794" text-anchor="start" letter-spacing="1.4" xml:space="preserve">ATTRIBUTES</text>
<text x="1152" y="430" font-size="17" font-weight="500" fill="#618794" text-anchor="start" xml:space="preserve">latency</text>
<text x="1538" y="430" font-size="17" font-weight="600" fill="#0b2026" text-anchor="end" xml:space="preserve">0.46 s</text>
<text x="1152" y="462" font-size="17" font-weight="500" fill="#618794" text-anchor="start" xml:space="preserve">status</text>
<text x="1538" y="462" font-size="17" font-weight="600" fill="#0b2026" text-anchor="end" xml:space="preserve">OK</text>
<text x="1152" y="494" font-size="17" font-weight="500" fill="#618794" text-anchor="start" xml:space="preserve">via</text>
<text x="1538" y="494" font-size="17" font-weight="600" fill="#0b2026" text-anchor="end" xml:space="preserve">managed MCP server</text>
<line x1="1152" y1="540" x2="1538" y2="540" stroke="#EEEDE9" stroke-width="1.5"/>
<text x="1152" y="574" font-size="17" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">Judges read these spans to grade tool choice,</text>
<text x="1152" y="599" font-size="17" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">not just the final answer.</text>
</svg>
</div>

The span types you will see most often in agent evaluation:

| Span type | What it records |
|---|---|
| `AGENT` | The agent's own reasoning and orchestration steps |
| `CHAT_MODEL` | Requests to a chat model and its responses |
| `TOOL` | Tool and function calls, with their inputs and results |
| `RETRIEVER` | Fetches from AI Search or another retriever |
| `CHAIN` | A sequence of operations run in order |

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<strong style="color: #0d47a1; font-size: 1.0em;">Full list of span types</strong>
<p style="margin: 8px 0 0 0; color: #333; font-size: 14px;">MLflow supports additional span types including <code>LLM</code>, <code>EMBEDDING</code>, <code>GUARDRAIL</code>, <code>PARSER</code>, <code>RERANKER</code>, <code>EVALUATOR</code>, <code>MEMORY</code>, <code>TASK</code>, <code>WORKFLOW</code>, and <code>UNKNOWN</code>. See the complete reference: (<a href="https://docs.databricks.com/aws/en/mlflow3/genai/tracing/span-concepts/" target="_blank" style="color:#1976d2;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/mlflow3/genai/tracing/span-concepts/" target="_blank" style="color:#1976d2;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/mlflow3/genai/tracing/span-concepts/" target="_blank" style="color:#1976d2;">GCP</a>)</p>
</div>


### D3. Why Tracing Matters for Evaluation

Certain evaluation judges, like `RetrievalSufficiency`, require traces to function. They analyze what was retrieved (not just the final response) to assess whether the retrieval system provided adequate information. Without traces, these sophisticated evaluations would be impossible.

Tracing also enables debugging by allowing you to inspect exactly what happened during a failed evaluation, identifying whether issues stem from retrieval quality, tool selection, or LLM reasoning.

Docs: (<a href="https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/concepts/scorers/" target="_blank">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/mlflow3/genai/eval-monitor/concepts/scorers" target="_blank">Azure</a> | <a href="https://docs.databricks.com/gcp/en/mlflow3/genai/eval-monitor/concepts/scorers/" target="_blank">GCP</a>)


## E. Production Monitoring and Unity Gateway

Evaluation doesn't stop at deployment. The same MLflow pieces you use offline also cover a deployed agent:

- **Traces from production**: the deployed agent logs a trace for every request to an MLflow experiment. In this course, the Airbnb assistant app on Databricks Apps logs to its own experiment, and the feedback users give in the app is saved on those traces.
- **Production monitoring**: register the same scorers you use offline so they run automatically on a sample of production traces, with `.register()` and `.start()`.
- **Unity Gateway**: every model call from the agent and from the judges goes through a Unity Gateway model service, so permissions, rate limits, and usage tracking apply to both.

Failures and edge cases you find in production become new rows in your evaluation dataset. The **Offline vs. Online Evaluation Strategies** lecture walks through that loop.

Production Monitoring (<a href="https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/" target="_blank">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/mlflow3/genai/eval-monitor/" target="_blank">Azure</a> | <a href="https://docs.databricks.com/gcp/en/mlflow3/genai/eval-monitor/" target="_blank">GCP</a>)

<div style="width: 100%; font-family: sans-serif;">

<div style="background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 16px 18px 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07);">

  <div style="height: 5px; border-radius: 5px; margin: -4px 0 12px; background: #FF5F46;"></div>

  <img src="https://files.training.databricks.com/binder/prod_main/genai-agent-evaluation-on-databricks-en_us-1.2.0/images/20261116T170304Z/Agent Evaluation on Databricks/Includes/images/genie-code.png" style="height: 64px; margin-bottom: 10px;">

  <div style="font-size: 15pt; color: #0b2026; line-height: 1.7; margin-bottom: 16px;">
    Want to learn more about MLflow's evaluation capabilities? Ask Genie Code. Click on the genie icon <img src="https://files.training.databricks.com/binder/prod_main/genai-agent-evaluation-on-databricks-en_us-1.2.0/images/20261116T170304Z/Agent Evaluation on Databricks/Includes/images/genie-icon.png" style="height: 32px; vertical-align: middle;"> and begin querying. For example, click the <strong>Copy</strong> button below and paste into <strong>Genie Code</strong>.
  </div>

  <div style="display: flex; align-items: center; gap: 10px; background: #F9F7F4; border: 1px solid #ddd; border-radius: 6px; padding: 10px 14px; font-size: 14pt; font-family: monospace; color: #0b2026;">
    <span id="genie-1-3" style="flex: 1;">How does mlflow.genai.evaluate() work and what parameters does it accept?</span>
    <button onclick="
      var text = document.getElementById('genie-1-3').innerText;
      var ta = document.createElement('textarea');
      ta.value = text;
      ta.style.position = 'fixed';
      ta.style.opacity = '0';
      document.body.appendChild(ta);
      ta.select();
      document.execCommand('copy');
      document.body.removeChild(ta);
      this.innerText = 'Copied!';
      var btn = this;
      setTimeout(function(){ btn.innerText = 'Copy'; }, 2000);
    " style="background: #FF5F46; color: white; border: none; border-radius: 4px; padding: 4px 12px; font-size: 13pt; cursor: pointer; white-space: nowrap;">Copy</button>
  </div>

</div>

</div>


&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
